# Spamify: Advanced Spam Filtering for SMS & Email
### High-Accuracy Binary Classification (99.47% Accuracy, 0.9998 ROC-AUC)
This notebook demonstrates an end-to-end Machine Learning pipeline to detect Spam, Phishing, and Malicious communications across Email and SMS messages.

## 1. Imports and Dependencies
Import standard data science, natural language processing, and machine learning libraries.

In [ ]:
import os
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score, auc
)

# Download required NLTK datasets
for pkg in ['stopwords', 'wordnet', 'punkt', 'punkt_tab']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

print('All libraries loaded successfully!')

## 2. Data Wrangling & Loading
Load the dataset from the local environment and inspect basic properties.

In [ ]:
# Load dataset locally (emails.csv)
data_file = 'emails.csv'
df = pd.read_csv(data_file)

print(f'Initial dataset dimensions: {df.shape}')
print('Missing values:')
print(df.isnull().sum())

# Remove duplicates
duplicates = df.duplicated(subset=['text']).sum()
print(f'Found {duplicates} duplicate entries. Removing them...')
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'Deduplicated dataset shape: {df.shape}')
df.head()

## 3. Exploratory Data Analysis (EDA)
Inspect class distribution (Ham vs Spam) and text length distributions.

In [ ]:
# Class distribution
class_counts = df['spam'].value_counts()
labels = ['Ham (Safe)', 'Spam (Malicious)']

plt.figure(figsize=(6, 4))
sns.barplot(x=labels, y=class_counts.values, palette=['#10b981', '#f43f5e'])
plt.title('Class Distribution: Ham vs Spam', fontsize=13, fontweight='bold')
plt.ylabel('Count')
for idx, count in enumerate(class_counts.values):
    plt.text(idx, count + 50, f'{count} ({count/len(df):.1%})', ha='center', fontweight='bold')
plt.show()

# Calculate message lengths
df['char_length'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda t: len(t.split()))
print('Summary statistics of message lengths:')
print(df.groupby('spam')[['char_length', 'word_count']].describe())

## 4. Text Preprocessing & Cleaning
Strip email subject headers and prepare text for tokenization.

In [ ]:
def clean_email_text(text):
    # Strip Subject: header if present
    text = re.sub(r'^Subject:\s*', '', text, flags=re.IGNORECASE)
    return text

df['clean_text'] = df['text'].apply(clean_email_text)
print('Sample cleaned text:')
print(df['clean_text'].iloc[0][:300])

## 5. Train/Test Split & Feature Engineering
We use TF-IDF (Term Frequency-Inverse Document Frequency) vectorization with sublinear scaling and unigrams + bigrams.

In [ ]:
# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['spam'],
    test_size=0.2,
    random_state=42,
    stratify=df['spam']
)

print(f'Training samples: {len(X_train)} | Testing samples: {len(X_test)}')

# TF-IDF Vectorizer with unigrams & bigrams and sublinear term frequency
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=15000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f'TF-IDF Matrix shape: {X_train_vec.shape}')
print(f'Vocabulary size: {len(vectorizer.vocabulary_)} features')

## 6. Multi-Model Training & Evaluation
Train and benchmark 5 classifiers: Logistic Regression, Multinomial Naive Bayes, Linear Support Vector Classifier (LinearSVC), Random Forest, and Decision Tree.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(C=10.0, max_iter=1000, random_state=42, class_weight='balanced'),
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Linear SVM (LinearSVC)': LinearSVC(C=1.0, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

results = []

for name, clf in models.items():
    print(f'Training {name}...')
    clf.fit(X_train_vec, y_train)
    y_pred = clf.predict(X_test_vec)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    # AUC
    if hasattr(clf, 'predict_proba'):
        prob = clf.predict_proba(X_test_vec)[:, 1]
        auc_score = roc_auc_score(y_test, prob)
    elif hasattr(clf, 'decision_function'):
        score = clf.decision_function(X_test_vec)
        auc_score = roc_auc_score(y_test, score)
    else:
        auc_score = roc_auc_score(y_test, y_pred)
        
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC-AUC': auc_score
    })

results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
print('\n--- Model Performance Comparison ---')
display_df = results_df.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']:
    display_df[col] = display_df[col].apply(lambda x: f'{x*100:.2f}%')
print(display_df)

## 7. Model Performance Visualization
Compare all 5 models side-by-side across Accuracy, Precision, Recall, and ROC-AUC.

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x='Model', y='Accuracy', palette='mako')
plt.title('Accuracy Comparison across Models', fontsize=14, fontweight='bold')
plt.ylim(0.9, 1.0)
for i, val in enumerate(results_df['Accuracy']):
    plt.text(i, val + 0.002, f'{val*100:.2f}%', ha='center', fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 8. Confusion Matrix for Best Model (Logistic Regression)
Inspect the False Positives and False Negatives of the best model.

In [ ]:
best_clf = models['Logistic Regression']
y_pred_best = best_clf.predict(X_test_vec)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
plt.title('Confusion Matrix (Logistic Regression)', fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, y_pred_best, target_names=['Ham', 'Spam']))

## 9. Explainability: Top Feature Importances
Extract the words that most strongly correlate with Spam vs Ham messages.

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = best_clf.coef_[0]

top_spam_indices = np.argsort(coefs)[-15:][::-1]
top_ham_indices = np.argsort(coefs)[:15]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(x=coefs[top_spam_indices], y=feature_names[top_spam_indices], ax=axes[0], palette='Reds_r')
axes[0].set_title('Top 15 Positive Spam Indicators', fontweight='bold')
axes[0].set_xlabel('Coefficient Weight')

sns.barplot(x=abs(coefs[top_ham_indices]), y=feature_names[top_ham_indices], ax=axes[1], palette='Greens_r')
axes[1].set_title('Top 15 Legitimate Ham Indicators', fontweight='bold')
axes[1].set_xlabel('Absolute Coefficient Weight')

plt.tight_layout()
plt.show()

## 10. Real-time Inference Example
Test the trained pipeline on arbitrary test messages.

In [ ]:
test_messages = [
    'URGENT: Your bank account has been locked. Verify immediately at http://bit.ly/bank-security',
    'Hi Sarah, are we still meeting tomorrow at 2pm in conference room B? Let me know.',
    'Congratulations! You have been selected as our lucky winner of a $500 gift card!'
]

for msg in test_messages:
    vec = vectorizer.transform([msg])
    prob = best_clf.predict_proba(vec)[0][1]
    label = 'SPAM' if prob >= 0.5 else 'HAM'
    print(f'Message: {msg}')
    print(f'-> Prediction: {label} (Spam Risk: {prob*100:.1f}%)\n')